# NLP Experiment 3 – Tweet sentiment analysis with TF-IDF (Sentiment140)
**Runtime:** Python 3 (CPU). Runtime → Run all.

**TF-IDF** weights each word by how often it appears in a tweet (TF) × how *rare* it is across all tweets (IDF),
so common words count less. Classifier: **Logistic Regression**. Labels: 0 = negative, 1 = positive.
Dataset: **Sentiment140** – 1.6 million tweets; we use a balanced sample of 100,000 for speed.

In [ ]:
!pip install -q kagglehub

In [ ]:
import os
import re
import kagglehub
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
from sklearn.metrics import classification_report, confusion_matrix

## Load Sentiment140 (no Kaggle login needed for this public dataset)

In [ ]:
path = kagglehub.dataset_download("kazanova/sentiment140")
file = os.path.join(path, "training.1600000.processed.noemoticon.csv")

df = pd.read_csv(file, encoding="latin-1", header=None,
                 names=["sentiment", "id", "date", "query", "user", "tweet"])
df["sentiment"] = df["sentiment"].replace({4: 1})       # 0 = negative, 4 -> 1 = positive

# balanced sample: 50,000 of each class
df = df.groupby("sentiment").sample(n=50000, random_state=42)
print(df["sentiment"].value_counts())
df[["sentiment", "tweet"]].head()

## Clean the tweets (URLs, @mentions, #, punctuation)

In [ ]:
def clean_tweet(text):
    text = text.lower()
    text = re.sub(r"http\S+|www\S+", " ", text)   # URLs
    text = re.sub(r"@\w+", " ", text)              # @mentions
    text = re.sub(r"#", "", text)                   # keep the hashtag word, drop '#'
    text = re.sub(r"[^a-z\s]", " ", text)          # punctuation, numbers, emojis
    text = re.sub(r"\s+", " ", text).strip()
    return text

df["clean"] = df["tweet"].apply(clean_tweet)
df = df[df["clean"] != ""]
df[["tweet", "clean"]].head()

## (a) TF-IDF features

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    df["clean"], df["sentiment"], test_size=0.2, random_state=42, stratify=df["sentiment"])

tfidf = TfidfVectorizer(max_features=20000, ngram_range=(1, 2))   # single words + word pairs
X_train_tfidf = tfidf.fit_transform(X_train)
X_test_tfidf = tfidf.transform(X_test)

print("TF-IDF matrix shape:", X_train_tfidf.shape)

## (b) Train Logistic Regression

In [ ]:
model = LogisticRegression(max_iter=1000)
model.fit(X_train_tfidf, y_train)
y_pred = model.predict(X_test_tfidf)

print(f"Accuracy : {accuracy_score(y_test, y_pred):.4f}")
print(f"Precision: {precision_score(y_test, y_pred):.4f}")
print(f"Recall   : {recall_score(y_test, y_pred):.4f}")
print(f"F1-score : {f1_score(y_test, y_pred):.4f}")
print(classification_report(y_test, y_pred, target_names=["Negative", "Positive"]))

## Visualise the results

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

cm = confusion_matrix(y_test, y_pred)
sns.heatmap(cm, annot=True, fmt="d", ax=axes[0],
            xticklabels=["Neg", "Pos"], yticklabels=["Neg", "Pos"])
axes[0].set_title("Confusion matrix")
axes[0].set_xlabel("Predicted")
axes[0].set_ylabel("Actual")

predicted = pd.Series(y_pred).map({0: "Negative", 1: "Positive"}).value_counts().reindex(["Negative", "Positive"])
axes[1].bar(predicted.index, predicted.values, color=["red", "green"])
axes[1].set_title("Predicted sentiment of test tweets")

plt.tight_layout()
plt.show()

In [ ]:
# words that push a tweet most towards negative / positive
words = tfidf.get_feature_names_out()
weights = model.coef_[0]
order = weights.argsort()
print("Most negative:", [words[i] for i in order[:10]])
print("Most positive:", [words[i] for i in order[-10:]])

## (c) Evaluate the sentiment of new product reviews / posts

In [ ]:
posts = [
    "I absolutely love this phone, the camera is amazing!",
    "Worst purchase ever, it broke after two days.",
    "Delivery was quick and the product works perfectly.",
    "Very disappointed and unhappy with the service.",
]
vectors = tfidf.transform([clean_tweet(p) for p in posts])
labels = model.predict(vectors)
probs = model.predict_proba(vectors)

for post, label, prob in zip(posts, labels, probs):
    print(f"{'Positive' if label == 1 else 'Negative'} ({max(prob) * 100:.1f}%) | {post}")